In [ ]:
# @title
# BLOQUE 1: Preparación y Análisis
!pip install -q mido midi2audio
!sudo apt-get install -q fluidsynth fluid-soundfont-gm -y

import mido
import os
from google.colab import files
from IPython.display import clear_output

clear_output()

In [ ]:
# @title
clear_output()

print("Por favor, sube tu archivo MIDI (.mid):")
subidos = files.upload()
ruta_midi = next(iter(subidos)) if subidos else None

def analizar_detalles_midi(ruta_archivo):
    if not ruta_archivo or not os.path.exists(ruta_archivo):
        print("Archivo no encontrado.")
        return

    mid = mido.MidiFile(ruta_archivo)
    canal_instrumento = {i: 0 for i in range(16)}
    teclas_unicas_por_inst = {}

    # Variables para el análisis absoluto
    todas_las_notas = set()
    total_pulsaciones = 0

    for pista in mid.tracks:
        for mensaje in pista:
            if mensaje.type == 'program_change':
                canal_instrumento[mensaje.channel] = mensaje.program
            elif mensaje.type == 'note_on' and mensaje.velocity > 0:
                # Registro absoluto
                total_pulsaciones += 1
                todas_las_notas.add(mensaje.note)

                # Registro por canal/instrumento
                canal = mensaje.channel
                if canal == 9:
                    nombre_instrumento = "Set de Percusión (Canal 10)"
                else:
                    num_programa = canal_instrumento[canal]
                    nombre_instrumento = f"Instrumento Programa {num_programa} (Canal {canal})"

                if nombre_instrumento not in teclas_unicas_por_inst:
                    teclas_unicas_por_inst[nombre_instrumento] = set()
                teclas_unicas_por_inst[nombre_instrumento].add(mensaje.note)

    print(f"\n--- Análisis Avanzado de: {ruta_archivo} ---")
    if not teclas_unicas_por_inst:
         print("No se encontraron notas en este archivo.")
         return

    # Imprimir análisis por canal
    for inst, teclas_set in teclas_unicas_por_inst.items():
        teclas = sorted(list(teclas_set))
        cantidad = len(teclas)
        min_nota = min(teclas)
        max_nota = max(teclas)
        diferencia = max_nota - min_nota
        octavas = sorted(list(set((nota // 12) - 1 for nota in teclas)))

        print(f"\n[{inst}]")
        print(f"  -> Teclas específicas (valores MIDI): {teclas}")
        print(f"  -> Cantidad total de teclas distintas: {cantidad}")
        print(f"  -> Nota Mínima: {min_nota} | Nota Máxima: {max_nota} | Diferencia: {diferencia} semitonos")
        print(f"  -> Octavas en las que toca: {octavas}")

    # Imprimir resumen global/absoluto
    if todas_las_notas:
        min_abs = min(todas_las_notas)
        max_abs = max(todas_las_notas)
        dif_abs = max_abs - min_abs
        octavas_abs = sorted(list(set((nota // 12) - 1 for nota in todas_las_notas)))

        print("\n" + "="*60)
        print("📊 RESUMEN GENERAL (ABSOLUTO) DEL ARCHIVO MIDI")
        print("="*60)
        print(f"  -> Mínima nota absoluta tocada: {min_abs}")
        print(f"  -> Máxima nota absoluta tocada: {max_abs}")
        print(f"  -> Distancia absoluta: {dif_abs} semitonos")
        print(f"  -> Total de veces que se presionó una tecla: {total_pulsaciones}")
        print(f"  -> Octavas totales usadas: {octavas_abs}")
        print("="*60 + "\n")

if ruta_midi:
    analizar_detalles_midi(ruta_midi)

In [ ]:
# @title
# BLOQUE 2: Escuchar el MIDI completo y por canales
from midi2audio import FluidSynth
from IPython.display import display, Audio
import os
import mido

clear_output()

ruta_soundfont = '/usr/share/sounds/sf2/FluidR3_GM.sf2'
sintetizador = FluidSynth(ruta_soundfont)

def escuchar_midi_y_canales(ruta_archivo):
    if not os.path.exists(ruta_archivo):
        print("Archivo no encontrado. Ejecuta el Bloque 1 primero.")
        return

    print(f"🎵 Generando audio completo para: {ruta_archivo}...")
    wav_completo = "completo.wav"
    sintetizador.midi_to_audio(ruta_archivo, wav_completo)
    display(Audio(wav_completo))
    print("-" * 50)

    mid = mido.MidiFile(ruta_archivo)
    canales_activos = set()
    for pista in mid.tracks:
        for msj in pista:
            if msj.type == 'note_on' and msj.velocity > 0:
                canales_activos.add(msj.channel)

    if not canales_activos:
        print("No se encontraron notas en el archivo.")
        return

    print("🎧 Generando audios separados por canal...")
    for canal_objetivo in canales_activos:
        nuevo_mid = mido.MidiFile()
        nuevo_mid.ticks_per_beat = mid.ticks_per_beat

        for pista in mid.tracks:
            nueva_pista = mido.MidiTrack()
            for msj in pista:
                if msj.is_meta or (hasattr(msj, 'channel') and msj.channel == canal_objetivo):
                    nueva_pista.append(msj.copy())
            if len(nueva_pista) > 0:
                nuevo_mid.tracks.append(nueva_pista)

        nombre_base = "Percusion" if canal_objetivo == 9 else f"Instrumento_Canal_{canal_objetivo}"
        archivo_mid_aislado = f"{nombre_base}.mid"
        archivo_wav = f"{nombre_base}.wav"

        nuevo_mid.save(archivo_mid_aislado)
        sintetizador.midi_to_audio(archivo_mid_aislado, archivo_wav)

        print(f"Pista aislada: {nombre_base}")
        display(Audio(archivo_wav))

if 'ruta_midi' in locals() and ruta_midi:
    escuchar_midi_y_canales(ruta_midi)
else:
    print("Ejecuta primero el Bloque 1 para cargar el archivo.")

In [ ]:
# @title
# BLOQUE 3: Conversión, Fusión y Exportación
import os
import re
import struct
from collections import defaultdict
import mido
import ipywidgets as w
from IPython.display import display
from google.colab import files

clear_output()

# --- Parámetros de Firmware ---
TPB = 12
NUM_TECLAS = 32
BPM_MIN, BPM_MAX = 40, 208
MAX_TITULO = 15
FLAG_ON, FLAG_REQ = 0x01, 0x02

def leer_midi(ruta):
    mid = mido.MidiFile(ruta)
    tempo = 500000
    compas = (4, 4)
    tempo_ok = compas_ok = False
    notas = defaultdict(list)

    for pista in mid.tracks:
        t = 0
        abiertas = defaultdict(list)
        for msj in pista:
            t += msj.time
            if msj.type == 'set_tempo' and not tempo_ok:
                tempo, tempo_ok = msj.tempo, True
            elif msj.type == 'time_signature' and not compas_ok:
                compas, compas_ok = (msj.numerator, msj.denominator), True
            elif msj.type == 'note_on' and msj.velocity > 0:
                abiertas[(msj.channel, msj.note)].append(t)
            elif msj.type in ('note_off', 'note_on'):
                pila = abiertas[(msj.channel, msj.note)]
                if pila:
                    inicio = pila.pop(0)
                    notas[msj.channel].append((inicio, t, msj.note))
    return mid, tempo, compas, notas

def elegir_base(notas_midi):
    if not notas_midi: return 60
    mediana = sorted(notas_midi)[len(notas_midi) // 2]
    mejor, mejor_clave = 0, None
    for base in range(0, 128 - NUM_TECLAS + 1):
        dentro = sum(1 for n in notas_midi if base <= n < base + NUM_TECLAS)
        clave = (-dentro, abs((base + (NUM_TECLAS - 1) / 2) - mediana))
        if mejor_clave is None or clave < mejor_clave:
            mejor, mejor_clave = base, clave
    return mejor

def plegar(nota, base):
    while nota < base: nota += 12
    while nota >= base + NUM_TECLAS: nota -= 12
    return nota

def convertir(ruta, canales_seleccionados, titulo=None):
    mid, tempo, compas, notas = leer_midi(ruta)
    ppq = mid.ticks_per_beat
    aviso = []

    lista_combinada = []
    for ch in canales_seleccionados:
        if ch in notas:
            lista_combinada.extend(notas[ch])

    if not lista_combinada:
        raise ValueError(f'Los canales seleccionados no tienen notas.')

    lista_combinada.sort(key=lambda x: (x[0], x[2]))

    bpm = round(mido.tempo2bpm(tempo))
    if not (BPM_MIN <= bpm <= BPM_MAX):
        bpm_orig = bpm
        bpm = max(BPM_MIN, min(BPM_MAX, bpm))
        aviso.append(f'BPM original {bpm_orig} fuera de rango, limitado a {bpm}.')

    def q(t): return int(round(t * TPB / ppq))

    lista_q = []
    for ini, fin, nota in lista_combinada:
        qi, qf = q(ini), q(fin)
        if qf <= qi: qf = qi + 1
        lista_q.append((qi, qf, nota))

    todas = [n for _, _, n in lista_q]
    base = elegir_base(todas)
    plegadas = 0
    notas_k = []

    for qi, qf, nota in lista_q:
        n2 = plegar(nota, base)
        if n2 != nota: plegadas += 1
        notas_k.append((qi, qf, n2 - base))

    eventos = []
    for qi, qf, k in notas_k:
        eventos.append((qi, 1, k, FLAG_ON | FLAG_REQ))
        eventos.append((qf, 0, k, 0))
    eventos.sort(key=lambda e: (e[0], e[1], e[2]))

    t0 = eventos[0][0]
    cuerpo = bytearray()
    previo = t0
    pasos = set()

    for tick, _, k, flags in eventos:
        delta = tick - previo
        if delta > 0xFFFF:
            raise ValueError(f'Silencio de {delta} ticks: supera el máximo (65535).')
        cuerpo += struct.pack('<BBH', k, flags, delta)  # (tecla, acción, delta), como lee el firmware
        if flags & FLAG_ON:
            pasos.add(tick)
        previo = tick

    n_eventos = len(eventos)
    n_pasos = len(pasos)

    if titulo is None:
        titulo = os.path.splitext(os.path.basename(ruta))[0]
    titulo_b = titulo.encode('ascii', 'ignore')[:MAX_TITULO].ljust(16, b'\0')
    header = struct.pack('<4sBBHBBBBHH16sH', b'APR1', 1, TPB, bpm, compas[0], compas[1], base, 0, n_eventos, n_pasos, titulo_b, 0)  # 34 B, como lee el firmware

    datos = bytes(header) + bytes(cuerpo)
    info = dict(bpm=bpm, compas=compas, base=base, notas=len(lista_q),
                plegadas=plegadas, eventos=n_eventos, pasos=n_pasos,
                bytes=len(datos), rango=(min(todas), max(todas)), avisos=aviso)
    return datos, info

def nombre_83(texto):
    limpio = re.sub(r'[^A-Za-z0-9]', '', texto).upper()
    return (limpio or 'CANCION')[:8]

def identificador_c(texto):
    ident = re.sub(r'[^A-Za-z0-9_]', '_', texto).lower().strip('_') or 'cancion'
    if ident[0].isdigit(): ident = '_' + ident
    return 'cancion_' + ident

def escribir_apr(datos, nombre):
    ruta = f'{nombre_83(nombre)}.apr'
    with open(ruta, 'wb') as f: f.write(datos)
    return ruta

def escribir_h(datos, nombre, titulo):
    ident = identificador_c(nombre)
    guarda = ident.upper() + '_H'
    lineas = ['    ' + ', '.join(f'0x{b:02X}' for b in datos[i:i+16]) for i in range(0, len(datos), 16)]
    cuerpo = ',\n'.join(lineas)
    texto = (f'/* {titulo} ({len(datos)} bytes) */\n#ifndef {guarda}\n#define {guarda}\n\n'
             f'#include <stdint.h>\n\nconst uint8_t {ident}[{len(datos)}] __attribute__((aligned(4))) = {{\n'
             f'{cuerpo}\n}};\n\n#define {ident.upper()}_SIZE {len(datos)}\n\n#endif\n')
    ruta = f'{ident}.h'
    with open(ruta, 'w') as f: f.write(texto)
    return ruta

def lanzar_interfaz_fusion(ruta_midi):
    if not os.path.exists(ruta_midi):
        print("Ejecuta el Bloque 1 para subir un MIDI.")
        return

    mid, tempo, compas, notas = leer_midi(ruta_midi)
    canales = [(ch, len([n for _,_,n in notas[ch]]), min([n for _,_,n in notas[ch]]), max([n for _,_,n in notas[ch]])) for ch in sorted(notas)]

    if not canales:
        print('El MIDI no tiene notas.')
        return

    opciones = [(f'Canal {c[0]} {"(Percusión)" if c[0]==9 else ""} - {c[1]} notas, rango {c[2]}..{c[3]}', c[0]) for c in canales]

    sel_canales = w.SelectMultiple(options=opciones, description='Canales:', layout=w.Layout(width='600px'))
    dd_fmt = w.Dropdown(options=[('.apr (SD)', '.apr'), ('.h (flash)', '.h'), ('Ambos', 'ambos')], value='ambos', description='Formato:')
    txt_tit = w.Text(value=os.path.splitext(os.path.basename(ruta_midi))[0][:MAX_TITULO], description='Título:', placeholder='máx. 15 caracteres')
    btn = w.Button(description='Generar y descargar', button_style='success')
    out = w.Output()

    def al_click(_):
        with out:
            out.clear_output()
            seleccion = list(sel_canales.value)
            if not seleccion:
                print("Por favor, selecciona al menos un canal.")
                return

            try:
                datos, info = convertir(ruta_midi, seleccion, txt_tit.value)
                sufijo = ''.join(map(str, seleccion))
                base_nombre = f"{os.path.splitext(os.path.basename(ruta_midi))[0]}_CH{sufijo}"
                tit = txt_tit.value or base_nombre

                archivos = []
                if dd_fmt.value in ('.apr', 'ambos'): archivos.append(escribir_apr(datos, base_nombre))
                if dd_fmt.value in ('.h', 'ambos'): archivos.append(escribir_h(datos, base_nombre, tit))

                print(f"BPM: {info['bpm']} | Base MIDI (tecla 0): {info['base']}")
                print(f"Notas totales unificadas: {info['notas']} (rango {info['rango'][0]}..{info['rango'][1]})")
                print(f"Plegadas por octava: {info['plegadas']} | Tamaño final: {info['bytes']} bytes")
                for a in archivos:
                    print(f'Descargando: {a}')
                    files.download(a)

            except Exception as e:
                print(f'Error: {e}')

    btn.on_click(al_click)
    display(w.VBox([
        w.Label("💡 Mantén presionado Ctrl (Windows) o Cmd (Mac) al hacer clic para seleccionar múltiples canales y fusionarlos."),
        sel_canales, dd_fmt, txt_tit, btn, out
    ]))

if 'ruta_midi' in locals() and ruta_midi:
    lanzar_interfaz_fusion(ruta_midi)
else:
    print("Ejecuta primero el Bloque 1 para cargar el archivo.")